# Step 1: Clone
increased timeout for retry to fectch the faild repos


In [ ]:
from __future__ import annotations
import csv, os, re, subprocess, time, base64, shutil
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from urllib.parse import urlparse

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT       = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3")
URL_LIST_CSV    = WORK_ROOT / "URL_List_RQ3.csv"
CLONE_ROOT      = WORK_ROOT / "Clone_retry_repos"

# ✅ Retry-only mode config
RETRY_FAILED_ONLY   = True
INPUT_MANIFEST_CSV  = WORK_ROOT / "clones_manifest_V1.csv"          # read failures from here
OUTPUT_MANIFEST_CSV = WORK_ROOT / "clones_manifest_retry_only.csv"  # write new manifest here

# Token file + keys
TOKENS_ENV_FILE = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
TOKEN_KEYS      = [f"GITHUB_TOKEN_{i}" for i in range(1, 7)]

# Optional: keep your PR refs option
FETCH_PR_REFS   = True

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
DEFAULT_TIMEOUT = 1800  # 30 minutes
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")

def load_env_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path}")
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and v and k not in os.environ:
            os.environ[k] = v

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def url_host_and_mode(url: str) -> Tuple[Optional[str], str]:
    u = url.strip()
    if re.match(r"^[^@]+@([^:]+):", u):
        host = u.split("@", 1)[1].split(":", 1)[0]
        return host, "ssh"
    if "://" in u:
        parsed = urlparse(u)
        host = (parsed.netloc or "").split("@")[-1].split(":")[0] or None
        return host, parsed.scheme.lower()
    return None, "other"

def git_auth_config_args(host: str, token: str) -> List[str]:
    basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
    return ["-c", f"http.https://{host}/.extraheader=Authorization: Basic {basic}"]

def sh(
    cmd: List[str],
    cwd: Optional[Path] = None,
    check: bool = True,
    capture: bool = True,
    timeout: Optional[int] = DEFAULT_TIMEOUT,
    auth_url: Optional[str] = None,
    token: Optional[str] = None
) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)

    if cmd and cmd[0] == "git" and token and auth_url:
        host, mode = url_host_and_mode(auth_url)
        if host and mode in ("https", "http"):
            cmd = ["git", *git_auth_config_args(host, token), *cmd[1:]]

    return subprocess.run(
        cmd, cwd=cwd, check=check,
        capture_output=capture, text=True,
        timeout=timeout, env=env
    )

def is_valid_git_repo(path: Path) -> bool:
    if not path.exists() or not (path / ".git").exists():
        return False
    cp = sh(["git", "rev-parse", "--is-inside-work-tree"], cwd=path, check=False, capture=True)
    return cp.returncode == 0 and (cp.stdout or "").strip().lower() == "true"

def load_failed_urls_from_manifest(manifest_csv: Path) -> List[str]:
    if not manifest_csv.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_csv}")
    failed: List[str] = []
    with manifest_csv.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            url = (row.get("repo_url") or "").strip()
            status = (row.get("status") or "").strip().lower()
            if url and status != "ok":
                failed.append(url)
    # de-dupe while preserving order
    seen = set()
    out = []
    for u in failed:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

class TokenRotator:
    def __init__(self, tokens: List[str]) -> None:
        self.tokens = tokens[:]
        self.i = 0

    def next(self) -> Tuple[int, str]:
        if not self.tokens:
            raise RuntimeError("No tokens available")
        idx = self.i % len(self.tokens)
        self.i += 1
        return (idx + 1, self.tokens[idx])

def is_retryable_auth_error(msg: str) -> bool:
    m = (msg or "").lower()
    needles = [
        "rate limit", "abuse detection", "too many requests", "http 429",
        "http 403", "403 forbidden", "http 401", "401 unauthorized",
        "authentication failed", "could not read username", "access denied",
        "fatal: unable to access", "remote: permission to"
    ]
    return any(n in m for n in needles)

def ensure_full_clone_no_submodules_no_lfs(
    url: str,
    dest_root: Path,
    fetch_pr_refs: bool,
    token: Optional[str] = None
) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    # ✅ If previous attempt left a broken folder, wipe it before retry
    if d.exists() and not is_valid_git_repo(d):
        log(f"[cleanup] Removing broken repo dir: {d}")
        shutil.rmtree(d, ignore_errors=True)

    if not (d.exists() and (d / ".git").exists()):
        sh(
            ["git", "clone", "--no-single-branch", "--no-recurse-submodules", "--tags", "--quiet", url, str(d)],
            capture=True,
            auth_url=url,
            token=token
        )
    else:
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(["git", "config", "core.longpaths", "true"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False, capture=True, auth_url=url, token=token)
    if cp.returncode == 0 and (cp.stdout or "").strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags", "--quiet"], cwd=d, capture=True, auth_url=url, token=token)
    else:
        sh(["git", "fetch", "--tags", "--quiet"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(
        ["git", "fetch", "origin", "--prune", "--tags",
         "+refs/heads/*:refs/remotes/origin/*", "--quiet"],
        cwd=d, check=False, capture=True, auth_url=url, token=token
    )

    if fetch_pr_refs:
        sh(
            ["git", "fetch", "origin",
             "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"],
            cwd=d, check=False, capture=True, auth_url=url, token=token
        )

    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

def clone_with_token_rotation(
    url: str,
    clone_root: Path,
    fetch_pr_refs: bool,
    rotator: Optional[TokenRotator],
    max_token_attempts: int = 6
) -> Tuple[Path, Dict[str, object]]:
    host, mode = url_host_and_mode(url)
    meta: Dict[str, object] = {"auth_mode": None, "token_slot": None, "attempts": 0}

    if mode == "ssh":
        meta["auth_mode"] = "ssh"
        meta["attempts"] = 1
        d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
        return d, meta

    if rotator and mode in ("https", "http") and host:
        meta["auth_mode"] = "https-token"
        last_err: Optional[Exception] = None
        tries = min(max_token_attempts, len(rotator.tokens))
        for _ in range(tries):
            slot, token = rotator.next()
            meta["attempts"] += 1
            meta["token_slot"] = slot
            try:
                d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=token)
                return d, meta
            except subprocess.CalledProcessError as e:
                err = (e.stderr or e.stdout or str(e))[:5000]
                last_err = e
                if is_retryable_auth_error(err):
                    log(f"Retryable auth/rate error for {url} using token slot {slot}; rotating token...")
                    continue
                raise
        if last_err:
            raise last_err
        raise RuntimeError("Clone failed after token rotation attempts")

    meta["auth_mode"] = "https-no-token"
    meta["attempts"] = 1
    d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
    return d, meta

# -----------------------------
# Main
# -----------------------------
# Load tokens
rotator: Optional[TokenRotator] = None
try:
    load_env_file(TOKENS_ENV_FILE)
    tokens = [os.environ.get(k) for k in TOKEN_KEYS]
    tokens = [t for t in tokens if t and t.strip()]
    if tokens:
        rotator = TokenRotator(tokens)
        log(f"Loaded {len(tokens)} GitHub token(s) from {TOKENS_ENV_FILE.name} (slots: 1..{len(tokens)})")
    else:
        log(f"No tokens found in {TOKENS_ENV_FILE.name}; proceeding without tokens.")
except Exception as e:
    log(f"Token file not loaded ({e}); proceeding without tokens.")

# ✅ Choose target URLs
if RETRY_FAILED_ONLY:
    target_urls = load_failed_urls_from_manifest(INPUT_MANIFEST_CSV)
    log(f"Retry mode: FAILED ONLY. Found {len(target_urls)} repo(s) to retry from: {INPUT_MANIFEST_CSV.name}")
else:
    assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"
    target_urls = []
    with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            url = (row.get("repo_url") or "").strip()
            if url:
                target_urls.append(url)
    log(f"Retry mode: ALL. Found {len(target_urls)} repo(s) from: {URL_LIST_CSV.name}")

rows, ok, fail = [], 0, 0
for url in target_urls:
    t0 = time.time()
    rec: Dict[str, object] = {
        "repo_url": url,
        "dir": None,
        "status": "unknown",
        "seconds": None,
        "total_commits": None,
        "error": "",
        "auth_mode": None,
        "token_slot": None,
        "attempts": None,
    }

    try:
        d, meta = clone_with_token_rotation(
            url=url,
            clone_root=CLONE_ROOT,
            fetch_pr_refs=FETCH_PR_REFS,
            rotator=rotator,
            max_token_attempts=6
        )
        rec["dir"] = str(d)
        rec["total_commits"] = get_total_commits(d)
        rec["status"] = "ok"
        rec.update(meta)
        ok += 1

        log(f"[ok] {url} -> {rec['dir']}  commits={rec['total_commits']}  "
            f"auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")

    except subprocess.CalledProcessError as e:
        rec["status"] = "error"
        rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
        fail += 1
        log(f"[error] {url}  auth={rec.get('auth_mode')} token_slot={rec.get('token_slot')} attempts={rec.get('attempts')}")

    except Exception as e:
        rec["status"] = "error"
        rec["error"]  = str(e)[:2000]
        fail += 1
        log(f"[error] {url}  ({e})")

    rec["seconds"] = round(time.time() - t0, 2)
    rows.append(rec)

# Write retry manifest (does not overwrite your original)
OUTPUT_MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["repo_url","dir","status","seconds","total_commits","error","auth_mode","token_slot","attempts"]
with OUTPUT_MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

log(f"Done. OK={ok}, FAIL={fail}. Manifest: {OUTPUT_MANIFEST_CSV}")


[2025-12-14 22:05:26] Loaded 6 GitHub token(s) from All_Tokens.env (slots: 1..6)
[2025-12-14 22:05:28] [ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\connectbot__connectbot  commits=4100  auth=https-token token_slot=1 attempts=1
[2025-12-14 22:05:29] [ok] https://github.com/ge0rg/aprsdroid -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\ge0rg__aprsdroid  commits=1211  auth=https-token token_slot=2 attempts=1
[2025-12-14 22:06:07] [ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\robolectric__robolectric  commits=21638  auth=https-token token_slot=3 attempts=1
[2025-12-14 22:06:36] [ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\opendocument-app__OpenDocument.droid  c